# Exploratory Data Analysis - Thesis USP

This notebook queries the BigQuery tables (`content`, `crux`, `pagespeed`) under the `thesisusp` dataset to perform exploratory data analysis.

In [2]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize the BigQuery client for project 'thesisusp'
client = bigquery.Client(project="thesisusp")

/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


## 1. Fetch Content Extraction Data

Big query só entende SQL por isso tem uma chamada a db no bigquery em SQL

In [3]:
query_content = """
SELECT * 
FROM `thesisusp.content`
"""
df_content = client.query(query_content).to_dataframe()
print("Registros de conteúdo carregados:", df_content.shape[0])
df_content.head()

/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Registros de conteúdo carregados: 2243


,run_str,sku,store,url,timestamp,category,title,meta_description,h1,description_word_count,...,schema_brand,schema_description,schema_aggregateRating,schema_image,schema_sku,schema_gtin,schema_offers,has_errors,extraction_blocked,robots_txt_status_code
0,20260701_1800,boticario-botik-serum-alta-potencia-vitamina-c...,Boticário (D2C),https://boticario.com.br/serum-de-alta-potenci...,2026-07-01 18:00:45.195929+00:00,skincare,Access Denied,NaN,Access Denied,12,...,False,False,False,False,False,False,False,False,False,<NA>
1,20260701_1800,boticario-botik-serum-alta-potencia-vitamina-c...,Mercado Livre,https://www.mercadolivre.com.br/o-boticario-bo...,2026-07-01 18:01:05.910795+00:00,skincare,Mercado Libre,NaN,NaN,36,...,False,False,False,False,False,False,False,False,True,403
2,20260701_1800,boticario-botik-serum-alta-potencia-vitamina-c...,Amazon Brasil,https://www.amazon.com.br/Botik-Vitamina-S%C3%...,2026-07-01 18:01:06.445103+00:00,skincare,Botik Vitamina C Sérum de Alta Potência 30ml O...,Compre online Botik Vitamina C Sérum de Alta P...,Botik Vitamina C Sérum de Alta Potência 30ml O...,5728,...,False,False,False,False,False,False,False,False,False,200
3,20260701_1800,iphone-17-pro,Apple Brasil,https://www.apple.com/br/shop/buy-iphone/iphon...,2026-07-01 18:00:10.271440+00:00,electronics,Comprar iPhone 17 Pro de 512 GB – Laranja-cósm...,Confira os novos iPhone 17 Pro e iPhone 17 Pro...,Comprar iPhone 17 Pro,2250,...,False,True,False,True,True,False,True,False,False,200
4,20260701_1800,iphone-17-pro,Vivo,https://store.vivo.com.br/apple-iphone-17-pro-...,2026-07-01 18:00:12.054710+00:00,electronics,Just a moment...,NaN,NaN,6,...,False,False,False,False,False,False,False,False,True,403


In [12]:
# Verificando quantos erros tivemos
print(f"Total de registros antes da limpeza: {len(df_content)}")
# Filtrando: Queremos apenas onde NÃO tem erro (== False) e NÃO foi bloqueado (== False)
df_content_clean = df_content[(df_content['has_errors'] == False) & (df_content['extraction_blocked'] == False)]
print(f"Total de registros após limpeza (URLs válidas): {len(df_content_clean)}")

Total de registros antes da limpeza: 2243
Total de registros após limpeza (URLs válidas): 1112


In [13]:
# Agrupando por Categoria e SKU, e calculando a Média, Mínimo e Máximo de palavras
resumo_palavras = df_content_clean.groupby(['category', 'sku'])['description_word_count'].agg(['mean', 'min', 'max', 'count']).reset_index()

# Renomeando as colunas para ficar bonitinho
resumo_palavras.columns = ['Categoria', 'Produto', 'Média de Palavras', 'Mínimo', 'Máximo', 'Qtd de Lojas']

display(resumo_palavras)


,Categoria,Produto,Média de Palavras,Mínimo,Máximo,Qtd de Lojas
0,electronics,iphone-17-pro,901.542969,98,2250,256
1,electronics,samsung-galaxy-a56-5g-256gb-preto-8gb-ram,168.385246,18,5407,122
2,electronics,samsung-galaxy-s26-512gb-12gb-ram-preto,882.5,18,12330,120
3,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,1981.054217,12,6158,166
4,skincare,la-roche-posay-pure-vitamin-c12-serum-30ml,38.5,12,65,166
5,skincare,natura-serum-intensivo-antioxidante-chronos-15...,1714.900621,12,6843,161
6,skincare,neutrogena-hydro-boost-water-gel-50g,240.247934,18,342,121


In [14]:
# Opção A: Comparar as Lojas lado a lado para o mesmo Produto
resumo_lojas = df_content_clean.groupby(['category', 'sku', 'store'])['description_word_count'].agg(['mean', 'min', 'max']).reset_index()

# Ordenando para ver quem escreve mais no topo
resumo_lojas = resumo_lojas.sort_values(by=['sku', 'mean'], ascending=[True, False])

display(resumo_lojas.head(15))


,category,sku,store,mean,min,max
8,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Amazon Brasil,3997.04878,18,6158
10,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Mercado Livre,101.0,101,101
9,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Boticário (D2C),12.0,12,12
1,electronics,iphone-17-pro,Apple Brasil,2229.835294,1393,2250
2,electronics,iphone-17-pro,Fastshop,488.0,488,488
3,electronics,iphone-17-pro,Kabum,365.0,365,365
0,electronics,iphone-17-pro,Americanas,98.0,98,98
12,skincare,la-roche-posay-pure-vitamin-c12-serum-30ml,La Roche-Posay Brasil,65.0,65,65
11,skincare,la-roche-posay-pure-vitamin-c12-serum-30ml,Droga Raia,12.0,12,12
13,skincare,natura-serum-intensivo-antioxidante-chronos-15...,Amazon Brasil,3526.961538,18,6843


In [16]:
# Analisando o tamanho da descrição e se a loja tem marcação de Preço e Review no Schema
resumo_seo = df_content_clean.groupby(['category', 'sku', 'store']).agg({
    'description_word_count': 'mean',
    'schema_price': 'max',  # Retorna True se a loja tem markup de preço
    'schema_aggregateRating': 'max' # Retorna True se a loja tem markup de reviews
}).reset_index()

resumo_seo = resumo_seo.sort_values(by=['sku', 'description_word_count'], ascending=[True, False])

display(resumo_seo.head(15))


,category,sku,store,description_word_count,schema_price,schema_aggregateRating
8,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Amazon Brasil,3997.04878,False,False
10,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Mercado Livre,101.0,True,True
9,skincare,boticario-botik-serum-alta-potencia-vitamina-c...,Boticário (D2C),12.0,False,False
1,electronics,iphone-17-pro,Apple Brasil,2229.835294,True,False
2,electronics,iphone-17-pro,Fastshop,488.0,True,False
3,electronics,iphone-17-pro,Kabum,365.0,True,False
0,electronics,iphone-17-pro,Americanas,98.0,True,False
12,skincare,la-roche-posay-pure-vitamin-c12-serum-30ml,La Roche-Posay Brasil,65.0,False,False
11,skincare,la-roche-posay-pure-vitamin-c12-serum-30ml,Droga Raia,12.0,False,False
13,skincare,natura-serum-intensivo-antioxidante-chronos-15...,Amazon Brasil,3526.961538,False,False


In [10]:
#df_content.info() 
#print(df_content.shape) 
#print(df_content.dtypes)

In [11]:
df_content.describe()


,description_word_count,llms_txt_status_code,schema_fields_count,robots_txt_status_code
count,2243.0,2159.0,2243.0,2107.0
mean,452.129291,424.369616,2.401694,318.059326
std,1418.488381,39.81338,4.057739,101.300545
min,0.0,403.0,0.0,200.0
25%,6.0,403.0,0.0,200.0
50%,18.0,403.0,0.0,403.0
75%,65.0,404.0,7.0,403.0
max,12330.0,503.0,10.0,429.0


o description word count poder ser uma soma. Essa soma pode me informar se eu tenho um 200 falso-positivo. (tenho que entender em que situações eu tenho um falso-positivo e se tenho um workaround para isto). 

Llms_txt_status_code nao faz sentido. Eu devo entender quantas veces o status aparece. porque? muitos consultores de geo estao vendendo que esse arquivo é essencial para geo. O Google nunca mencionou isto em nenhuma documentacao.

Schema fields count. ok mas tenho que decompor estes números. 

O das llms, vale para o robots porém sabemos que é essencial para SEO.

In [ ]:
# Query para calcular o total de requisições, total de bloqueios e a taxa percentual de bloqueio por loja
query_bloqueios = """
SELECT 
    store, 
    COUNT(*) as total_requisicoes, 
    SUM(CASE WHEN extraction_blocked THEN 1 ELSE 0 END) as requisicoes_bloqueadas,
    ROUND(SUM(CASE WHEN extraction_blocked THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) as taxa_bloqueio_percentual
FROM `thesisusp.content`
GROUP BY store
ORDER BY taxa_bloqueio_percentual DESC
"""


df_bloqueios = client.query(query_bloqueios).to_dataframe()
df_bloqueios


/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,store,total_requisicoes,requisicoes_bloqueadas,taxa_bloqueio_percentual
0,Magazine Luiza,118,118,100.00
1,Vivo,177,177,100.00
2,Cosmetis,59,59,100.00
3,Mercado Livre,236,235,99.58
4,Fastshop,59,53,89.83
5,Samsung,118,59,50.00
6,Amazon Brasil,295,104,35.25
7,Apple Brasil,61,0,0.00
8,Kabum,118,0,0.00
9,La Roche-Posay Brasil,59,0,0.00


In [20]:
query_skus_bloqueio = """
SELECT 
    store, 
    sku,
    CASE WHEN extraction_blocked THEN 'sim' ELSE 'nao' END as bloqueado,
    COUNT(*) as total_linhas
FROM `thesisusp.content`
GROUP BY store, sku, extraction_blocked
ORDER BY store, sku, bloqueado
"""
df_skus_bloqueio = client.query(query_skus_bloqueio).to_dataframe()
df_skus_bloqueio

/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,store,sku,bloqueado,total_linhas
0,Amazon Brasil,boticario-botik-serum-alta-potencia-vitamina-c...,nao,59
1,Amazon Brasil,motorola-edge-60-5g-512gb-azul-marinho-12gb-ram,nao,28
2,Amazon Brasil,motorola-edge-60-5g-512gb-azul-marinho-12gb-ram,sim,31
3,Amazon Brasil,natura-serum-intensivo-antioxidante-chronos-15...,nao,57
4,Amazon Brasil,natura-serum-intensivo-antioxidante-chronos-15...,sim,2
5,Amazon Brasil,neutrogena-hydro-boost-water-gel-50g,nao,24
6,Amazon Brasil,neutrogena-hydro-boost-water-gel-50g,sim,35
7,Amazon Brasil,samsung-galaxy-s26-512gb-12gb-ram-preto,nao,23
8,Amazon Brasil,samsung-galaxy-s26-512gb-12gb-ram-preto,sim,36
9,Americanas,iphone-17-pro,nao,59


In [21]:
query_amazon_por_sku = """
SELECT 
    sku, 
    COUNT(*) as total_requisicoes, 
    SUM(CASE WHEN extraction_blocked THEN 1 ELSE 0 END) as bloqueadas,
    ROUND(SUM(CASE WHEN extraction_blocked THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) as taxa_bloqueio_pct
FROM `thesisusp.content`
WHERE store = 'Amazon Brasil'
GROUP BY sku
ORDER BY taxa_bloqueio_pct DESC
"""

client.query(query_amazon_por_sku).to_dataframe()


/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,sku,total_requisicoes,bloqueadas,taxa_bloqueio_pct
0,samsung-galaxy-s26-512gb-12gb-ram-preto,59,36,61.02
1,neutrogena-hydro-boost-water-gel-50g,59,35,59.32
2,motorola-edge-60-5g-512gb-azul-marinho-12gb-ram,59,31,52.54
3,natura-serum-intensivo-antioxidante-chronos-15...,59,2,3.39
4,boticario-botik-serum-alta-potencia-vitamina-c...,59,0,0.00


In [22]:
query_amazon_por_run = """
SELECT 
    run_str, 
    COUNT(*) as total_requisicoes, 
    SUM(CASE WHEN extraction_blocked THEN 1 ELSE 0 END) as bloqueadas
FROM `thesisusp.content`
WHERE store = 'Amazon Brasil'
GROUP BY run_str
ORDER BY run_str
LIMIT 20
"""

client.query(query_amazon_por_run).to_dataframe()


/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,run_str,total_requisicoes,bloqueadas
0,20260604_1319,5,2
1,20260604_1800,5,3
2,20260605_0000,5,0
3,20260605_1300,5,2
4,20260605_1800,5,1
5,20260606_0000,5,2
6,20260606_1300,5,2
7,20260606_1800,5,2
8,20260607_0000,5,2
9,20260607_1300,5,2


In [23]:
query_assinatura_bloqueios = """
SELECT 
    title, 
    COUNT(*) as ocorrencias, 
    AVG(description_word_count) as media_palavras,
    -- Identificamos o tipo de resposta da Amazon
    CASE 
        WHEN title = 'Amazon.com.br' AND AVG(description_word_count) < 30 THEN 'Tela de CAPTCHA (Bloqueio)'
        WHEN title IS NULL OR title = '' THEN 'Sem Resposta / Timeout'
        WHEN title LIKE '%503%' THEN 'Erro 503 (Serviço Indisponível)'
        WHEN AVG(description_word_count) < 10 THEN 'Bloqueio Parcial (Página sem conteúdo)'
        ELSE 'Sucesso (Página do Produto)'
    END as status_analise
FROM `thesisusp.content`
WHERE store = 'Amazon Brasil'
GROUP BY title
ORDER BY ocorrencias DESC
"""

client.query(query_assinatura_bloqueios).to_dataframe()


/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,ocorrencias,media_palavras,status_analise
0,Amazon.com.br,106,18.000000,Tela de CAPTCHA (Bloqueio)
1,Botik Vitamina C Sérum de Alta Potência 30ml O...,43,5895.372093,Sucesso (Página do Produto)
2,"Celular Samsung Galaxy S26 Ultra 5G, 512GB, 12...",42,1221.928571,Sucesso (Página do Produto)
3,Neutrogena Hidratante Facial Hydro Boost Water...,34,3.000000,Bloqueio Parcial (Página sem conteúdo)
4,Sérum Intensivo Antioxidante Chronos - 15 ml |...,33,6568.909091,Sucesso (Página do Produto)
5,"Smartphone Samsung Galaxy A56 5G 256GB, 8GB RA...",18,1.000000,Bloqueio Parcial (Página sem conteúdo)
6,"Smartphone Samsung Galaxy A56 5G 128GB, 8GB RA...",8,2.000000,Bloqueio Parcial (Página sem conteúdo)
7,"Smartphone Samsung Galaxy A56 5G 128GB, 8GB RA...",6,900.833333,Sucesso (Página do Produto)
8,\n 503 - Erro de serviço indisponível\n,2,0.000000,Erro 503 (Serviço Indisponível)
9,NaN,2,0.000000,Sem Resposta / Timeout


In [24]:
query_assinaturas_todas = """
SELECT 
    store,
    title, 
    COUNT(*) as ocorrencias, 
    ROUND(AVG(description_word_count), 1) as media_palavras,
    -- Classificação inteligente das respostas
    CASE 
        WHEN title = 'Amazon.com.br' AND AVG(description_word_count) < 30 THEN 'Amazon CAPTCHA'
        WHEN title = 'Access Denied' THEN 'Bloqueio Ativo (WAF Access Denied)'
        WHEN title = 'Just a moment...' THEN 'Bloqueio Cloudflare Challenge'
        WHEN title = 'Mercado Libre' THEN 'Bloqueio Mercado Livre'
        WHEN title IS NULL OR title = '' THEN 'Sem Resposta (Timeout/Vazio)'
        WHEN title LIKE '%503%' THEN 'Erro 503 (Serviço Indisponível)'
        WHEN AVG(description_word_count) < 15 THEN 'Bloqueio Parcial (Página de Produto Vazia)'
        ELSE 'Sucesso (Página Completa)'
    END as status_diagnostico
FROM `thesisusp.content`
GROUP BY store, title
ORDER BY store, ocorrencias DESC
"""

df_assinaturas = client.query(query_assinaturas_todas).to_dataframe()
df_assinaturas


/opt/anaconda3/envs/tcc_usp/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,store,title,ocorrencias,media_palavras,status_diagnostico
0,Amazon Brasil,Amazon.com.br,106,18.0,Amazon CAPTCHA
1,Amazon Brasil,Botik Vitamina C Sérum de Alta Potência 30ml O...,43,5895.4,Sucesso (Página Completa)
2,Amazon Brasil,"Celular Samsung Galaxy S26 Ultra 5G, 512GB, 12...",42,1221.9,Sucesso (Página Completa)
3,Amazon Brasil,Neutrogena Hidratante Facial Hydro Boost Water...,34,3.0,Bloqueio Parcial (Página de Produto Vazia)
4,Amazon Brasil,Sérum Intensivo Antioxidante Chronos - 15 ml |...,33,6568.9,Sucesso (Página Completa)
5,Amazon Brasil,"Smartphone Samsung Galaxy A56 5G 256GB, 8GB RA...",18,1.0,Bloqueio Parcial (Página de Produto Vazia)
6,Amazon Brasil,"Smartphone Samsung Galaxy A56 5G 128GB, 8GB RA...",8,2.0,Bloqueio Parcial (Página de Produto Vazia)
7,Amazon Brasil,"Smartphone Samsung Galaxy A56 5G 128GB, 8GB RA...",6,900.8,Sucesso (Página Completa)
8,Amazon Brasil,\n 503 - Erro de serviço indisponível\n,2,0.0,Erro 503 (Serviço Indisponível)
9,Amazon Brasil,NaN,2,0.0,Sem Resposta (Timeout/Vazio)


In [ ]:
import requests
from bs4 import BeautifulSoup

# URL da Vivo (que sempre retorna bloqueio no BigQuery)
url_teste = "https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico" 

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    # fiz a requisicao local
    response = requests.get(url_teste, headers=headers, timeout=15)
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Extraí o título e a quantidade de palavras
    title = soup.title.string if soup.title else "Sem título"
    word_count = len(soup.body.get_text(strip=True).split()) if soup.body else 0
    
    print("--- RESULTADO DA EXTRAÇÃO LOCAL (Seu Computador) ---")
    print(f"Status HTTP: {response.status_code}")
    print(f"Título da Página: '{title}'")
    print(f"Quantidade de palavras: {word_count}")
    
except Exception as e:
    print("Erro ao tentar conectar:", str(e))


--- RESULTADO DA EXTRAÇÃO LOCAL (Seu Computador) ---
Status HTTP: 200
Título da Página: 'Vivo - Loja Oficial'
Quantidade de palavras: 133


In [ ]:
print(f"Título: {title}")
print("\n--- Início do Conteúdo Textual ---")
if soup.body:
    print(soup.body.get_text(strip=True)[:400])
else:
    print("Corpo vazio")


Título: Vivo - Loja Oficial

--- Início do Conteúdo Textual ---
Skip to HeaderSkip to Main ContentSkip to FooterAcesseCarrinhoMinha contaMeus pedidosMeus endereçosMeus dadosAjudaTrocas, devoluções e cancelamentoSairVoltarTodos departamentosCelularesÁudioTV e VídeoAcessóriosOfertasPlanosEasy Lite com CelularesPlanosVivo FibraPós PagoControlePré-pagoServiços DigitaisVivo Casa 5GConsórcioTroque seu planoVoltarCelularAcessóriosAcessóriosCapaPelículaCabos e Carrega


In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def extrair_como_humano():
    async with async_playwright() as p:
        # Lança o navegador Chromium visível na tela (headless=False)
        browser = await p.chromium.launch(headless=False)
        
        # Cria uma página simulando uma tela de computador normal - nao funcionou 
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800}
        )
        page = await context.new_page()
        
        url_vivo = "https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico"
        print(f"Acessando: {url_vivo}")
        
        # Acessa o site e aguarda o carregamento da rede ficar ocioso. 
        await page.goto(url_vivo, wait_until="networkidle")
        
        # Espera mais 5 segundos extras para garantir que o Javascript do preço carregou
        await page.wait_for_timeout(5000)
        
        # Pega o título oficial da página e o texto visível completo do corpo (body)
        title = await page.title()
        body_text = await page.locator("body").inner_text()
        
        # Fecha o navegador
        await browser.close()
        
        # Conta a quantidade de palavras
        word_count = len(body_text.split())
        
        print("\n--- RESULTADO COM PLAYWRIGHT (Simulador de Humano) ---")
        print(f"Título: {title}")
        print(f"Quantidade de palavras extraídas: {word_count}")
        print("\n--- Início do Texto Extraído ---")
        print(body_text[:500])

# Como estamos rodando no Jupyter (que já possui um loop de eventos ativo),
# executamos a função assíncrona usando await direto:
await extrair_como_humano()


Acessando: https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico

--- RESULTADO COM PLAYWRIGHT (Simulador de Humano) ---
Título: Vivo - Loja Oficial
Quantidade de palavras extraídas: 149

--- Início do Texto Extraído ---
Skip to Header
Skip to Main Content
Skip to Footer
Todos departamentos
Celulares
Áudio
TV e Vídeo
Acessórios
Ofertas
Planos 
Em até 21x sem juros
Entrega em todo Brasil
Frete grátis em toda loja
Compra Segura

Formas de pagamento

Essa loja utiliza cookies
A Vivo utiliza cookies de sessão e cookies persistentes para melhorar a sua experiência no site. Ao continuar navegando você concorda com a nossa política de privacidade
Continuar e fechar
Mais detalhes
Celulares 
iPhone
Samsung
Motorola
Acess


In [28]:
import asyncio
from playwright.async_api import async_playwright

async def extrair_com_firefox():
    async with async_playwright() as p:
        print("Lançando Firefox para simular humano...")
        # Lançamos o Firefox (geralmente passa mais fácil por proteções)
        browser = await p.firefox.launch(headless=False)
        
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:120.0) Gecko/20100101 Firefox/120.0",
            viewport={"width": 1280, "height": 800},
            locale="pt-BR",
            timezone_id="America/Sao_Paulo"
        )
        page = await context.new_page()
        
        url_vivo = "https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico"
        print(f"Acessando: {url_vivo}")
        
        # Acessa e espera 8 segundos para dar tempo do JS renderizar a tela
        try:
            await page.goto(url_vivo, wait_until="load", timeout=30000)
            await page.wait_for_timeout(8000)
            
            title = await page.title()
            body_text = await page.locator("body").inner_text()
            word_count = len(body_text.split())
            
            print("\n--- RESULTADO COM FIREFOX ---")
            print(f"Título: {title}")
            print(f"Palavras extraídas: {word_count}")
            print("\n--- Início do Texto Extraído ---")
            print(body_text[:500])
            
        except Exception as e:
            print("Erro durante o carregamento da página:", str(e))
            
        finally:
            await browser.close()

# Executa no Jupyter
await extrair_com_firefox()


Lançando Firefox para simular humano...
Acessando: https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico

--- RESULTADO COM FIREFOX ---
Título: Vivo - Loja Oficial
Palavras extraídas: 115

--- Início do Texto Extraído ---
Skip to Header
Skip to Main Content
Skip to Footer
Todos departamentos
Celulares
Áudio
TV e Vídeo
Acessórios
Ofertas
Planos 
Em até 21x sem juros
Entrega em todo Brasil
Frete grátis em toda loja
Compra Segura

Formas de pagamento

Mais detalhes
Celulares 
iPhone
Samsung
Motorola
Acessórios 
Ovvi
i2GO
Planos 
Planos
Pós Pago
Controle
Acessibilidade de Aparelhos
TV e Vídeo 
Smart TVs
Streaming
Áudio 
Fones de ouvido
Caixas de som
Casa inteligente 
Assistente virtual
Lâmpadas e interruptores
Câmera


In [31]:
import asyncio
from playwright.async_api import async_playwright

async def extrair_com_firefox():
    async with async_playwright() as p:
        print("Lançando Firefox para simular humano...")
        browser = await p.firefox.launch(headless=False)
        
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:120.0) Gecko/20100101 Firefox/120.0",
            viewport={"width": 1280, "height": 800},
            locale="pt-BR",
            timezone_id="America/Sao_Paulo"
        )
        page = await context.new_page()
        
        # URL da Apple para teste
        url_teste = "https://www.apple.com/br/"
        print(f"Acessando: {url_teste}")
        
        try:
            # Acessa e espera carregar
            await page.goto(url_teste, wait_until="load", timeout=30000)
            await page.wait_for_timeout(8000)
            
            title = await page.title()
            body_text = await page.locator("body").inner_text()
            word_count = len(body_text.split())
            
            print("\n--- RESULTADO COM FIREFOX ---")
            print(f"Título: {title}")
            print(f"Palavras extraídas: {word_count}")
            print("\n--- Início do Texto Extraído ---")
            print(body_text[:500])
            
        except Exception as e:
            print("Erro durante o carregamento da página:", str(e))
            
        finally:
            await browser.close()

# Executa no Jupyter
await extrair_com_firefox()



Lançando Firefox para simular humano...
Acessando: https://www.apple.com/br/

--- RESULTADO COM FIREFOX ---
Título: Apple (Brasil)
Palavras extraídas: 571

--- Início do Texto Extraído ---
Apple
Apple
Loja
Mac
iPad
iPhone
Apple Watch
AirPods
TV e Casa
Entretenimento
Acessórios
Suporte
0
+
AirPods Pro 3

O melhor Cancelamento Ativo de Ruído


do mundo em fones intra‑auriculares.1

Saiba mais
Comprar
iPhone

Conheça a nova geração do iPhone.

Saiba mais
Comprar iPhone
Apple para a faculdade

Mac e iPad. Brilhantes em qualquer disciplina.

Saiba mais
Apple Sports

Cada goooool em tempo real.

Baixe grátis
MacBook Air

Agora com a potência do M5.

Saiba mais
Comprar
iPad Air

Agora co


In [ ]:
import asyncio
from playwright.async_api import async_playwright
# Importamos a classe Stealth
from playwright_stealth import Stealth

async def extrair_com_stealth():
    async with async_playwright() as p:
        print("Lançando navegador com disfarce Stealth...")
        # Lança o Chromium (motor do Chrome)
        browser = await p.chromium.launch(headless=False)
        
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800},
            locale="pt-BR",
            timezone_id="America/Sao_Paulo"
        )
        page = await context.new_page()
        
        # APLICAÇÃO CORRETA DO STEALTH - ERREI NA ANTERIOR
        await Stealth().apply_stealth_async(page)
        
        # URL da Vivo
        url_vivo = "https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico"
        print(f"Acessando de forma oculta: {url_vivo}")
        
        try:
            # Acessa e aguarda o carregamento - revisar pq me perdi aqui
            await page.goto(url_vivo, wait_until="load", timeout=40000)
            await page.wait_for_timeout(10000) # Espera 10s para carregar bem
            
            title = await page.title()
            body_text = await page.locator("body").inner_text()
            word_count = len(body_text.split())
            
            print("\n--- RESULTADO COM PLAYWRIGHT STEALTH ---")
            print(f"Título: {title}")
            print(f"Palavras extraídas: {word_count}")
            print("\n--- Início do Texto Extraído ---")
            print(body_text[:500])
            
        except Exception as e:
            print("Erro durante o carregamento da página:", str(e))
            
        finally:
            await browser.close()

# Executa no Jupyter
await extrair_com_stealth()


Lançando navegador com disfarce Stealth...
Acessando de forma oculta: https://store.vivo.com.br/apple-iphone-17-pro-512gb-laranja-cosmico

--- RESULTADO COM PLAYWRIGHT STEALTH ---
Título: Vivo - Loja Oficial
Palavras extraídas: 115

--- Início do Texto Extraído ---
Skip to Header
Skip to Main Content
Skip to Footer
Todos departamentos
Celulares
Áudio
TV e Vídeo
Acessórios
Ofertas
Planos 
Em até 21x sem juros
Entrega em todo Brasil
Frete grátis em toda loja
Compra Segura

Formas de pagamento

Mais detalhes
Celulares 
iPhone
Samsung
Motorola
Acessórios 
Ovvi
i2GO
Planos 
Planos
Pós Pago
Controle
Acessibilidade de Aparelhos
TV e Vídeo 
Smart TVs
Streaming
Áudio 
Fones de ouvido
Caixas de som
Casa inteligente 
Assistente virtual
Lâmpadas e interruptores
Câmera


## 2. Fetch CrUX Performance Data

In [8]:
query_crux = """
SELECT * 
FROM `thesisusp.crux`
LIMIT 100
"""
df_crux = client.query(query_crux).to_dataframe()
print(df_content.shape)
print(len(df_content))

(100, 28)
100


## 3. Fetch PageSpeed Performance Data

In [4]:
query_pagespeed = """
SELECT * 
FROM `thesisusp.pagespeed`
LIMIT 100
"""
df_pagespeed = client.query(query_pagespeed).to_dataframe()
df_pagespeed.head()

/Users/evapaula/mba_thesis/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,run_str,sku,store,url,timestamp,category,mobile_score,mobile_cls,mobile_ttfb,mobile_fcp,mobile_inp,mobile_lcp,mobile_error,desktop_score,desktop_error
0,20260608_0000,boticario-botik-serum-alta-potencia-vitamina-c...,Boticário (D2C),https://boticario.com.br/serum-de-alta-potenci...,2026-06-08 00:38:12.894159+00:00,skincare,32.0,1.0,875.0,1528.0,515.0,1840.0,NaN,60.0,NaN
1,20260608_0000,boticario-botik-serum-alta-potencia-vitamina-c...,Mercado Livre,https://www.mercadolivre.com.br/o-boticario-bo...,2026-06-08 00:39:17.966596+00:00,skincare,53.0,0.0,1169.0,1887.0,183.0,2274.0,NaN,47.0,NaN
2,20260608_0000,boticario-botik-serum-alta-potencia-vitamina-c...,Amazon Brasil,https://www.amazon.com.br/Botik-Vitamina-S%C3%...,2026-06-08 00:40:18.502551+00:00,skincare,43.0,12.0,929.0,1542.0,168.0,2026.0,NaN,83.0,NaN
3,20260608_0000,iphone-17-pro,Apple Brasil,https://www.apple.com/br/shop/buy-iphone/iphon...,2026-06-08 00:03:47.296316+00:00,electronics,51.0,0.0,1389.0,2747.0,179.0,4092.0,NaN,43.0,NaN
4,20260608_0000,iphone-17-pro,Vivo,https://store.vivo.com.br/apple-iphone-17-pro-...,2026-06-08 00:05:47.543027+00:00,electronics,NaN,NaN,NaN,NaN,NaN,NaN,"HTTPSConnectionPool(host='www.googleapis.com',...",NaN,"HTTPSConnectionPool(host='www.googleapis.com',..."
